# Unleashed v1.2 — Python 3.13
Isi `BRANCH` untuk menguji branch lain (mis. `perf/phase1`); `main` = versi utama.
UI tampil di bawah sel *Jalankan Unleashed* (proxy Colab); link gradio.live di log tetap bisa dipakai.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q --upgrade pip

In [ ]:
#@title Clone & install
BRANCH = "main"  #@param {type:"string"}
!git clone -b {BRANCH} https://github.com/antorio/unleashed.git /content/unleashed
%cd /content/unleashed
!git log --oneline -1
!mv config_colab.yaml config.yaml
!pip install -q -r requirements.txt python-multipart


In [ ]:
import sys, numpy, torch, onnxruntime as ort
print('python', sys.version.split()[0], '| numpy', numpy.__version__, '| torch', torch.__version__)
print('cuda', torch.cuda.is_available(), '|', ort.get_available_providers())
assert 'CUDAExecutionProvider' in ort.get_available_providers()
import insightface, cv2, gradio
print('insightface', insightface.__version__, '| cv2', cv2.__version__, '| gradio', gradio.__version__)

In [ ]:
#@title Jalankan Unleashed
# UI muncul DI BAWAH sel ini lewat proxy Colab (jaringan Google) begitu server siap.
# Proxy Colab tidak bisa dibuka di tab baru: fitur itu sudah dimatikan Colab
# (pembaruan keamanan browser), jadi hanya bekerja di dalam bingkai ini.
# Link gradio.live di log tetap aktif sebagai cadangan (mis. untuk upload besar).
# Hentikan: tombol stop sel ini.
from google.colab import userdata, output
import os, re, subprocess, sys, time
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['NO_ALBUMENTATIONS_UPDATE'] = '1'
os.environ['PYTHONUNBUFFERED'] = '1'
LOG = '/content/run.log'
proc = subprocess.Popen([sys.executable, 'run.py'], cwd='/content/unleashed',
                        stdout=open(LOG, 'w'), stderr=subprocess.STDOUT)
log, head, port = open(LOG, errors='replace'), '', None
try:
    while proc.poll() is None:
        new = log.read()
        print(new, end='')
        if port is None:
            head += new
            m = re.search(r'Running on local URL:\s+https?://[^:/\s]+:(\d+)', head)
            if m:
                port = int(m.group(1))
                output.serve_kernel_port_as_iframe(port, height=900)
        time.sleep(1)
except KeyboardInterrupt:
    proc.terminate()
print(log.read(), end='')


## Bandingkan dua hasil render (opsional)
Jalankan setelah server dihentikan. Isi dua path video hasil (mis. render `main` vs render branch dari klip & setting yang sama).

In [ ]:
#@title Compare renders
A = "/content/drive/MyDrive/c/hasil_main.mp4"  #@param {type:"string"}
B = "/content/drive/MyDrive/c/hasil_branch.mp4"  #@param {type:"string"}
!python tools/compare_renders.py "{A}" "{B}"


## Uji: render berulang (opsional)
Hentikan dulu sel *Jalankan Unleashed*. Merender klip netral pendek (dari foto contoh insightface) beberapa kali, ER off dan on, berurutan dan tiga sekaligus, lalu membandingkan tiap frame. Salin baris-baris hasilnya (terutama `RESULT:`) ke Claude. ±5–10 menit; hanya menulis ke `/content/unleashed_test`.

In [ ]:
#@title Repeat render
!cd /content/unleashed && python tools/repeat_render.py

## Uji: di mana ER berbeda (opsional)
Hentikan dulu sel *Jalankan Unleashed*. Dua render ER per setup (default / deterministic / heuristic), mencatat tiap panggilan model LivePortrait, lalu menunjuk di mana kedua render mulai berbeda. Salin semua baris hasilnya ke Claude. ±5 menit; hanya menulis ke `/content/unleashed_test`.

In [ ]:
#@title ER trace
!cd /content/unleashed && python tools/er_trace.py

## Uji: langkah mana yang membuat frame berbeda (opsional)
Hentikan dulu sel *Jalankan Unleashed*. ER off, 4 render berurutan + 3 sekaligus; untuk tiap frame yang berbeda dari render pertama, menunjuk langkah pertama yang berbeda. Salin semua baris hasilnya ke Claude. ±5 menit; hanya menulis ke `/content/unleashed_test`.

In [ ]:
#@title Render trace
!cd /content/unleashed && git pull -q && python tools/render_trace.py

## Uji: model salah jawab saat dipakai beberapa thread (opsional)
Hentikan dulu sel *Jalankan Unleashed*. Swap & mask model, 8 thread, 4 cara pemanggilan; menghitung output yang salah. Salin semua baris hasilnya ke Claude. ±2–3 menit; tidak menulis apa pun.

In [ ]:
#@title ORT race
!cd /content/unleashed && git pull -q && python tools/ort_race.py